# 💻 Laboratório de Aquecimento - Lab 17: Moldando o Tempo (Séries Temporais)

**Objetivo:** Compreender a mecânica do tempo no Pandas. Você aprenderá a transformar registros diários barulhentos em agregações mensais limpas e a criar a nossa "Máquina do Tempo" (Lags) para ensinar o passado ao algoritmo de Machine Learning.

## 🔥 Aquecimento 1: Indexação Temporal e Agregação (`resample`)
Em séries temporais, a data não é apenas mais uma coluna, ela é o **Índice** do nosso DataFrame. O primeiro passo é transformar o DataFrame comum em uma série indexada e usar a função `.resample()` para "esmagar" os dados diários em faturamentos mensais.

**Sua Tarefa:** Execute o código abaixo para gerar nossos dados de vendas diárias. Em seguida, converta a data em índice e agrupe os dados por mês.

In [1]:
import pandas as pd
import numpy as np

# Gerando dados simulados de vendas diárias durante 3 meses (Jan a Mar)
np.random.seed(42)
datas = pd.date_range(start='2026-01-01', end='2026-03-31', freq='D')
vendas = np.random.randint(100, 500, size=len(datas))

df_vendas = pd.DataFrame({'Data_Venda': datas, 'Vendas_Diarias': vendas})

print("--- Primeiras 3 linhas (Dados Diários) ---")
display(df_vendas.head(3))

# ESCREVA SEU CÓDIGO AQUI: 

# 1. Garantir que a coluna temporal é do tipo datetime (Boa prática)
df_vendas['Data_Venda'] = pd.to_datetime(df_vendas['Data_Venda'])

# 2. Transformar a coluna de Data no Índice oficial do DataFrame
df_temporal = df_vendas.set_index('Data_Venda')

# 3. A Agregação Dinâmica: Use o .resample('M') para agrupar por Mês, somando as vendas
df_mensal = df_temporal.resample('M').sum() # 'M' significa fechamento Mensal

print("\n--- Fechamento Agregado (Dados Mensais) ---")
display(df_mensal)

--- Primeiras 3 linhas (Dados Diários) ---


,Data_Venda,Vendas_Diarias
0,2026-01-01,202
1,2026-01-02,448
2,2026-01-03,370



--- Fechamento Agregado (Dados Mensais) ---


/var/folders/7_/mlzr2t1j7rn6n62c72wwnrmh0000gn/T/ipykernel_7045/330734980.py:23: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_mensal = df_temporal.resample('M').sum() # 'M' significa fechamento Mensal


,Vendas_Diarias
Data_Venda,
2026-01-31,9697
2026-02-28,8371
2026-03-31,9761


## 🔥 Aquecimento 2: A Máquina do Tempo (`shift` e Lags)
Agora que temos o faturamento fechado mês a mês, nós temos um problema preditivo: um algoritmo lê os dados linha por linha. Como o modelo vai prever as vendas de Março se ele não consegue "olhar para cima" e ver quanto vendemos em Fevereiro? 

Nós criamos um **Lag** (Deslocamento). Vamos empurrar os dados do passado para a linha do presente usando a função `.shift()`.

**Sua Tarefa:** Use o `df_mensal` criado no passo anterior para criar as colunas do passado.

In [2]:
# ESCREVA SEU CÓDIGO AQUI:

# 1. Criando o Lag 1 (O mês anterior): Desloque a coluna 'Vendas_Diarias' em 1 posição para baixo
df_mensal['Vendas_Mes_Anterior'] = df_mensal['Vendas_Diarias'].shift(1)

# 2. Criando o Lag 2 (Dois meses atrás): Desloque a mesma coluna original em 2 posições para baixo
df_mensal['Vendas_2_Meses_Atras'] = df_mensal['Vendas_Diarias'].shift(2)

print("--- DataFrame Final com os Preditores do Passado ---")
display(df_mensal)

--- DataFrame Final com os Preditores do Passado ---


,Vendas_Diarias,Vendas_Mes_Anterior,Vendas_2_Meses_Atras
Data_Venda,,,
2026-01-31,9697,NaN,NaN
2026-02-28,8371,9697.0,NaN
2026-03-31,9761,8371.0,9697.0


### 🧠 Reflexão do Aquecimento:
Observe atentamente o `df_mensal` gerado no final do Aquecimento 2:
1. **O Alinhamento Perfeito:** Na linha correspondente a Março, você agora tem as vendas exatas de Fevereiro na coluna `Vendas_Mes_Anterior` e as vendas exatas de Janeiro na coluna `Vendas_2_Meses_Atras`. O algoritmo agora pode ler a linha de Março e ter todo o histórico do trimestre!
2. **O Efeito Colateral (`NaN`):** Repare que Janeiro ganhou `NaN` (nulos). Por quê? Porque Janeiro foi o primeiro mês da nossa base de dados; não existe um "mês anterior" a ele para ser deslocado. Ao trabalhar com Séries Temporais, gerar nulos no início do histórico é o preço obrigatório que se paga para criar *Lags*.

---
**🚀 Próximo Passo:** Se você compreendeu que a "Data" deve ser o Índice (`set_index`), como esmagar períodos (`resample`) e como criar variáveis do passado com o (`shift`), você está 100% preparado para abrir o caderno oficial **`lab_17_ts.ipynb`** e aplicar essa engenharia temporal para descobrir tendências e sazonalidades no nosso dataset oficial de Vendas!